# Brain Tumor Classification — Basic RNN Version
**Converted from CNN to Basic RNN (SimpleRNN)**

**Key idea:** Images are reshaped into sequences — each row of pixels (150 pixels × 3 channels = 450 features) becomes one timestep. So a 150×150 image becomes a sequence of 150 timesteps.

In [ ]:
# CELL 1: Upload Dataset
from google.colab import files
uploaded = files.upload()

In [ ]:
# CELL 2: Extract Archive
import zipfile
import os

with zipfile.ZipFile('archive (7).zip', 'r') as zip_ref:
    zip_ref.extractall('archive_data')

print(os.listdir('archive_data'))

In [ ]:
# CELL 3: Check Dataset Structure
print("\nDataset structure:")
for root, dirs, files in os.walk('archive_data'):
    level = root.replace('archive_data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            print(f'{subindent}{file}')

In [ ]:
# CELL 4: Setup Data Generators
# ============================================================
# NOTE: ImageDataGenerator is kept the same.
# The reshape from (H, W, C) -> (H, W*C) happens inside
# a custom generator wrapper defined in the next cell.
# ============================================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 150
BATCH_SIZE = 32

train_dir = '/content/archive_data/Training'
test_dir  = '/content/archive_data/Testing'

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("\nClass indices:", train_generator.class_indices)
print(f"Number of classes: {len(train_generator.class_indices)}")

In [ ]:
# CELL 5: RNN Generator Wrapper
# ============================================================
# RNNs expect input shape: (batch, timesteps, features)
# We convert each image (150, 150, 3) -> (150, 450)
# i.e., 150 timesteps, each timestep = one row of pixels
#       flattened across width and channels (150 * 3 = 450)
# ============================================================
import numpy as np

def rnn_generator(generator):
    """Wraps an ImageDataGenerator to reshape images for RNN input."""
    for images, labels in generator:
        # images: (batch, 150, 150, 3)  ->  rnn_input: (batch, 150, 450)
        batch_size = images.shape[0]
        rnn_input = images.reshape(batch_size, IMG_SIZE, IMG_SIZE * 3)
        yield rnn_input, labels

print(f"RNN input shape per sample: ({IMG_SIZE}, {IMG_SIZE * 3})")
print(f"  -> {IMG_SIZE} timesteps (one per image row)")
print(f"  -> {IMG_SIZE * 3} features per timestep (width x channels)")

In [ ]:
# CELL 6: Display Sample Images (unchanged)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle('Sample Training Images', fontsize=16)

images, labels = next(train_generator)

for i, ax in enumerate(axes.flat):
    if i < 9:
        ax.imshow(images[i])
        class_name = list(train_generator.class_indices.keys())[np.argmax(labels[i])]
        ax.set_title(f'Class: {class_name}')
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# CELL 7: Custom Accuracy Callback (unchanged)
import tensorflow as tf

class AccuracyCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if logs.get('accuracy') > 0.98:
            print(f"\nReached 98% accuracy at epoch {epoch+1}, stopping training!")
            self.model.stop_training = True

print("Custom callback defined successfully!")

In [ ]:
# CELL 8: Build Model A — Simple Basic RNN
# ============================================================
# ORIGINAL (CNN):                  REPLACED WITH (RNN):
#   Conv2D(32)  + MaxPooling2D  ->  SimpleRNN(32, return_sequences=True)
#   Conv2D(64)  + MaxPooling2D  ->  SimpleRNN(64, return_sequences=True)
#   Conv2D(128) + MaxPooling2D  ->  SimpleRNN(128)  [last layer, no return_sequences]
#   Flatten()                   ->  removed (RNN already outputs a vector)
#   Dense(128)  + Dropout(0.5)  ->  kept the same
#   Dense(num_classes)          ->  kept the same
# ============================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout

def build_model_A_RNN(input_shape=(IMG_SIZE, IMG_SIZE * 3), num_classes=3):
    model = Sequential([
        # Layer 1: replaces Conv2D(32) + MaxPooling2D
        SimpleRNN(32, return_sequences=True, activation='tanh',
                  input_shape=input_shape),

        # Layer 2: replaces Conv2D(64) + MaxPooling2D
        SimpleRNN(64, return_sequences=True, activation='tanh'),

        # Layer 3: replaces Conv2D(128) + MaxPooling2D + Flatten
        SimpleRNN(128, return_sequences=False, activation='tanh'),

        # Dense layers — same as original
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

num_classes = len(train_generator.class_indices)
model_A_rnn = build_model_A_RNN(num_classes=num_classes)
print("\nModel A (Basic RNN) Architecture:")
model_A_rnn.summary()

In [ ]:
# CELL 9: Build Model B — Deep Basic RNN
# ============================================================
# ORIGINAL (Deep CNN):             REPLACED WITH (Deep RNN):
#   Conv2D(32)  + MaxPooling2D  ->  SimpleRNN(32,  return_sequences=True)
#   Conv2D(64)  + MaxPooling2D  ->  SimpleRNN(64,  return_sequences=True)
#   Conv2D(128) + MaxPooling2D  ->  SimpleRNN(128, return_sequences=True)
#   Conv2D(128) + MaxPooling2D  ->  SimpleRNN(128, return_sequences=True)
#   Conv2D(256) + MaxPooling2D  ->  SimpleRNN(256, return_sequences=False)
#   Flatten()                   ->  removed
#   Dense(512) + Dropout(0.5)   ->  kept the same
#   Dense(256) + Dropout(0.5)   ->  kept the same
#   Dense(num_classes)          ->  kept the same
# ============================================================
def build_model_B_RNN(input_shape=(IMG_SIZE, IMG_SIZE * 3), num_classes=3):
    model = Sequential([
        # Layer 1: replaces Conv2D(32) + MaxPooling2D
        SimpleRNN(32,  return_sequences=True, activation='tanh',
                  input_shape=input_shape),

        # Layer 2: replaces Conv2D(64) + MaxPooling2D
        SimpleRNN(64,  return_sequences=True, activation='tanh'),

        # Layer 3: replaces Conv2D(128) + MaxPooling2D
        SimpleRNN(128, return_sequences=True, activation='tanh'),

        # Layer 4: replaces Conv2D(128) + MaxPooling2D
        SimpleRNN(128, return_sequences=True, activation='tanh'),

        # Layer 5: replaces Conv2D(256) + MaxPooling2D + Flatten
        SimpleRNN(256, return_sequences=False, activation='tanh'),

        # Dense layers — same as original
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

model_B_rnn = build_model_B_RNN(num_classes=num_classes)
print("\nModel B (Deep Basic RNN) Architecture:")
model_B_rnn.summary()

In [ ]:
# CELL 10: Train Model A — Simple RNN
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model_A_rnn.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_A = EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True, verbose=1
)
checkpoint_A = ModelCheckpoint(
    'Model_A_RNN_best.keras',
    monitor='val_accuracy', save_best_only=True, verbose=1
)
accuracy_cb_A = AccuracyCallback()

print("\n" + "="*60)
print("TRAINING MODEL A — SIMPLE RNN")
print("="*60)

history_A_rnn = model_A_rnn.fit(
    rnn_generator(train_generator),
    steps_per_epoch=len(train_generator),
    epochs=20,
    validation_data=rnn_generator(validation_generator),
    validation_steps=len(validation_generator),
    callbacks=[early_stop_A, checkpoint_A, accuracy_cb_A],
    verbose=1
)

print("\nModel A (Simple RNN) training completed!")

In [ ]:
# CELL 11: Train Model B — Deep RNN
model_B_rnn.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_B = EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True, verbose=1
)
checkpoint_B = ModelCheckpoint(
    'Model_B_RNN_best.keras',
    monitor='val_accuracy', save_best_only=True, verbose=1
)
accuracy_cb_B = AccuracyCallback()

print("\n" + "="*60)
print("TRAINING MODEL B — DEEP RNN")
print("="*60)

history_B_rnn = model_B_rnn.fit(
    rnn_generator(train_generator),
    steps_per_epoch=len(train_generator),
    epochs=20,
    validation_data=rnn_generator(validation_generator),
    validation_steps=len(validation_generator),
    callbacks=[early_stop_B, checkpoint_B, accuracy_cb_B],
    verbose=1
)

print("\nModel B (Deep RNN) training completed!")

In [ ]:
# CELL 12: Plot Training History
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history_A_rnn.history['accuracy'],     label='Model A RNN - Train')
plt.plot(history_A_rnn.history['val_accuracy'], '--', label='Model A RNN - Val')
plt.plot(history_B_rnn.history['accuracy'],     label='Model B RNN - Train')
plt.plot(history_B_rnn.history['val_accuracy'], '--', label='Model B RNN - Val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_A_rnn.history['loss'],     label='Model A RNN - Train')
plt.plot(history_A_rnn.history['val_loss'], '--', label='Model A RNN - Val')
plt.plot(history_B_rnn.history['loss'],     label='Model B RNN - Train')
plt.plot(history_B_rnn.history['val_loss'], '--', label='Model B RNN - Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# CELL 13: Evaluate All Models on Test Set
print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

models = {
    'Model A - Simple RNN': model_A_rnn,
    'Model B - Deep RNN':   model_B_rnn,
}

results = {}
for name, model in models.items():
    print(f"\nEvaluating {name}...")
    loss, accuracy = model.evaluate(
        rnn_generator(test_generator),
        steps=len(test_generator),
        verbose=0
    )
    results[name] = {'loss': loss, 'accuracy': accuracy}
    print(f"  Test Loss:     {loss:.4f}")
    print(f"  Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# CELL 14: Results Comparison Table
print("\n" + "="*60)
print("FINAL RESULTS COMPARISON")
print("="*60)
print("\n{:<25} {:<25} {:<15}".format('Model', 'Test Accuracy', 'Test Loss'))
print("-" * 65)
for name, metrics in results.items():
    print("{:<25} {:<25} {:<15}".format(
        name,
        f"{metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)",
        f"{metrics['loss']:.4f}"
    ))

In [ ]:
# CELL 15: Visualize Results (Bar Chart)
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(results.keys())
accuracies  = [results[n]['accuracy'] * 100 for n in model_names]
losses      = [results[n]['loss']           for n in model_names]

x     = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy (%)', color='skyblue')
ax2   = ax.twinx()
bars2 = ax2.bar(x + width/2, losses,    width, label='Loss',         color='coral')

ax.set_xlabel('Models', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax.set_title('Test Set Performance — Basic RNN Models', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=10, ha='right')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(True, alpha=0.3)

for bar1, bar2 in zip(bars1, bars2):
    ax.text(bar1.get_x() + bar1.get_width()/2., bar1.get_height(),
            f'{bar1.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
    ax2.text(bar2.get_x() + bar2.get_width()/2., bar2.get_height(),
             f'{bar2.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# CELL 16: Find Best Model
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_accuracy   = results[best_model_name]['accuracy']

print("\n" + "="*60)
print("BEST PERFORMING MODEL")
print("="*60)
print(f"\nModel:         {best_model_name}")
print(f"Test Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"Test Loss:     {results[best_model_name]['loss']:.4f}")

In [ ]:
# CELL 17: Sample Predictions from Best Model
best_model  = models[best_model_name]
class_names = list(train_generator.class_indices.keys())

# Get one batch and reshape for RNN
test_images, test_labels = next(test_generator)
rnn_input   = test_images.reshape(test_images.shape[0], IMG_SIZE, IMG_SIZE * 3)
predictions = best_model.predict(rnn_input)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'Sample Predictions — {best_model_name}', fontsize=16)

for i, ax in enumerate(axes.flat):
    if i < 6:
        ax.imshow(test_images[i])
        true_label = class_names[np.argmax(test_labels[i])]
        pred_label = class_names[np.argmax(predictions[i])]
        confidence = np.max(predictions[i]) * 100
        color = 'green' if true_label == pred_label else 'red'
        ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
                     color=color, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# CELL 18: Discussion and Analysis
print("\n" + "="*60)
print("DISCUSSION — CNN vs BASIC RNN")
print("="*60)
print("""
HOW THE CONVERSION WORKS:
  Each image (150 x 150 x 3) is reshaped into (150, 450).
  The RNN sees 150 timesteps, one per row of the image.
  Each timestep has 450 features (150 pixels x 3 colour channels).

LAYER-BY-LAYER MAPPING:
  CNN Layer               ->  RNN Equivalent
  Conv2D(32)+MaxPool      ->  SimpleRNN(32,  return_sequences=True)
  Conv2D(64)+MaxPool      ->  SimpleRNN(64,  return_sequences=True)
  Conv2D(128)+MaxPool     ->  SimpleRNN(128, return_sequences=True)
  Conv2D(128)+MaxPool     ->  SimpleRNN(128, return_sequences=True)  [Model B only]
  Conv2D(256)+MaxPool     ->  SimpleRNN(256, return_sequences=False) [Model B only]
  Flatten()               ->  removed  (RNN last layer already outputs a vector)
  Dense / Dropout         ->  unchanged

ACTIVATION:
  SimpleRNN uses tanh by default (its natural activation).
  CNNs used relu — kept relu only in the Dense layers.

TRAINING WRAPPER:
  rnn_generator() wraps the ImageDataGenerator to reshape
  each batch on-the-fly. steps_per_epoch / validation_steps
  are passed explicitly because the generator is now a plain
  Python generator (not a Keras Sequence).

KNOWN LIMITATION OF BasicRNN:
  SimpleRNN suffers from vanishing gradients on long sequences.
  150 timesteps is relatively long, so accuracy may be lower
  than the CNN baseline. LSTM or GRU would handle this better,
  but this notebook keeps to Basic RNN as requested.
""")
print("\n" + "="*60)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("="*60)